# nb10 — Backbone-Freeze Fine-Tuning

Task 2.1: fine-tune the poisoned model with backbone frozen (3 modes) and report proxy scores.
The proxy score = `suppression_gain - 10 * preservation_loss` mirrors the maCADD A=10 asymmetry.

**Freeze modes tested:**
- `heads_only`: freeze backbone (ResNet+FPN), train cls+reg heads
- `all_but_cls_head`: freeze backbone + reg head, train classification subnet only
- `lastFPN_plus_heads`: freeze ResNet only, train FPN + heads

**Reference (from Phase 1):** suppression_ref=0.603, preservation_ref=0.721

**Expected:** at least one freeze mode beats the nb01 unfreeze baseline on proxy.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import copy
import json
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from tqdm import tqdm

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUT_DIR          = Path("/kaggle/working")

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

# Solver (same as baseline for fair comparison)
UNLEARN_LR    = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE    = 4
SEED          = 42   # fixed seed for reproducibility across modes

# Proxy harness calibration (from Phase 1)
SUPPRESSION_REF  = 0.603
PRESERVATION_REF = 0.721
PRESERVE_WEIGHT  = 10.0

# Probe generation (replicated from nb03 for self-contained proxy harness)
N_PROBES       = 80
SKY_MEAN_U16   = 8000
STREAK_PEAK    = 45000
PROBES_DIR     = OUT_DIR / "probes"

print(f"Competition data: {COMP_ROOT}")

## Core pipeline functions

In [ ]:
def build_cfg_base(weights_path, output_dir, score_thresh=0.2, seed=42):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                        = str(weights_path)
    cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
    cfg.SEED = seed
    if output_dir:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        dataset_dict["image"]     = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


def compute_iou(box_a, box_b):
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    area_a = (box_a[2]-box_a[0])*(box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0])*(box_b[3]-box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def supp_score(predictor, unlearn_dir, iou_thresh=0.2):
    """Mean max-conf over poison boxes across 20 unlearn images. Lower=better."""
    with open(Path(unlearn_dir) / "annotations_coco.json") as f:
        coco = json.load(f)
    img_to_ann = {a["image_id"]: a for a in coco["annotations"]}
    vals = []
    for im_info in coco["images"]:
        ann = img_to_ann.get(im_info["id"])
        if ann is None: vals.append(0.0); continue
        bx, by, bw, bh = ann["bbox"]
        pbox = [bx, by, bx+bw, by+bh]
        im  = load_for_inference(Path(unlearn_dir) / im_info["file_name"])
        out = predictor(im)["instances"].to("cpu")
        best = max(
            (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
             if compute_iou([x1,y1,x2,y2], pbox) >= iou_thresh),
            default=0.0
        )
        vals.append(best)
    return float(np.mean(vals))


def pres_score(predictor, probes_dir):
    """Mean max-conf over annotated probe streaks. Higher=better."""
    with open(Path(probes_dir) / "probes_coco.json") as f:
        coco = json.load(f)
    img_to_anns = {}
    for a in coco["annotations"]:
        img_to_anns.setdefault(a["image_id"], []).append(a)
    vals = []
    for im_info in coco["images"]:
        for ann in img_to_anns.get(im_info["id"], []):
            bx, by, bw, bh = ann["bbox"]
            sbox = [bx, by, bx+bw, by+bh]
            im   = load_for_inference(Path(probes_dir) / im_info["file_name"])
            out  = predictor(im)["instances"].to("cpu")
            best = max(
                (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
                 if compute_iou([x1,y1,x2,y2], sbox) >= 0.2),
                default=0.0
            )
            vals.append(best)
    return float(np.mean(vals)) if vals else 0.0


def proxy(supp_now, pres_now, supp_ref=SUPPRESSION_REF, pres_ref=PRESERVATION_REF, w=PRESERVE_WEIGHT):
    """proxy = suppression_gain - w * preservation_loss."""
    gain = supp_ref - supp_now
    loss = max(0.0, pres_ref - pres_now)
    return gain - w * loss, gain, loss

## Generate synthetic preservation probes (deterministic, SEED=42)

In [ ]:
PROBES_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

def draw_streak(img_u16, x0, y0, x1, y1, sigma=3.0, peak=45000):
    dx, dy = x1-x0, y1-y0
    length = math.hypot(dx, dy)
    if length < 1: return None
    pad = int(sigma*3)+1
    bx0,by0 = max(0,min(x0,x1)-pad), max(0,min(y0,y1)-pad)
    bx1,by1 = min(IMG_W,max(x0,x1)+pad), min(IMG_H,max(y0,y1)+pad)
    ys,xs = np.mgrid[by0:by1, bx0:bx1]
    t = ((xs-x0)*dx+(ys-y0)*dy)/(length*length)
    tc = np.clip(t,0,1)
    dist = np.sqrt((xs-x0-tc*dx)**2+(ys-y0-tc*dy)**2)
    g = np.exp(-0.5*(dist/sigma)**2)
    taper = np.where(t<0,np.exp(-2*t**2),np.where(t>1,np.exp(-2*(t-1)**2),1.0))
    img_u16[by0:by1,bx0:bx1] = np.clip(
        img_u16[by0:by1,bx0:bx1].astype(np.float32)+g*taper*peak,0,65535).astype(np.uint16)
    return [bx0,by0,bx1-bx0,by1-by0]

coco_imgs, coco_anns, ann_id = [], [], 1
for idx in range(N_PROBES):
    bg = rng.normal(SKY_MEAN_U16, 300, (IMG_H,IMG_W)).clip(0,65535).astype(np.uint16)
    n_s = 1 if idx < int(N_PROBES*0.7) else 2
    bboxes = []
    for _ in range(n_s):
        bw,bh = int(rng.integers(20,81)), int(rng.integers(20,81))
        angle = rng.uniform(0, math.pi)
        cx = int(rng.integers(10+bw//2, IMG_W-10-bw//2))
        cy = int(rng.integers(10+bh//2, IMG_H-10-bh//2))
        hl = math.hypot(bw,bh)/2
        x0,y0 = int(cx-hl*math.cos(angle)), int(cy-hl*math.sin(angle))
        x1,y1 = int(cx+hl*math.cos(angle)), int(cy+hl*math.sin(angle))
        b = draw_streak(bg, x0, y0, x1, y1)
        if b: bboxes.append(b)
    fname = f"probe_{idx:04d}.png"
    cv2.imwrite(str(PROBES_DIR/fname), bg)
    iid = idx+1
    coco_imgs.append({"id":iid,"file_name":fname,"height":IMG_H,"width":IMG_W})
    for b in bboxes:
        coco_anns.append({"id":ann_id,"image_id":iid,"category_id":1,
                          "bbox":[float(v) for v in b],"area":float(b[2]*b[3]),"iscrowd":0})
        ann_id += 1

with open(PROBES_DIR/"probes_coco.json","w") as f:
    json.dump({"images":coco_imgs,"annotations":coco_anns,
               "categories":[{"id":1,"name":"streak"}]}, f)
print(f"Generated {N_PROBES} probes, {len(coco_anns)} annotations")

## Freeze logic + FreezeTrainer

In [ ]:
# Freeze-mode prefix definitions for Detectron2 RetinaNet R50-FPN
# backbone.bottom_up.* = ResNet  |  backbone.fpn_* = FPN  |  head.* = detection heads
FREEZE_PREFIXES = {
    "heads_only": [
        "backbone.",          # freeze entire backbone (ResNet + FPN)
    ],
    "all_but_cls_head": [
        "backbone.",          # freeze entire backbone
        "head.bbox_subnet.",  # freeze box regression subnet
        "head.bbox_pred.",
    ],
    "lastFPN_plus_heads": [
        "backbone.bottom_up.",  # freeze ResNet only; FPN + heads are trainable
    ],
}

_FREEZE_MODE = None  # module-level var read by FreezeTrainer.build_optimizer


def apply_freeze(model, mode):
    """Freeze parameters whose names start with any prefix in FREEZE_PREFIXES[mode]."""
    prefixes = FREEZE_PREFIXES[mode]
    n_frozen = n_train = 0
    for name, param in model.named_parameters():
        if any(name.startswith(p) for p in prefixes):
            param.requires_grad_(False)
            n_frozen += param.numel()
        else:
            param.requires_grad_(True)
            n_train += param.numel()
    print(f"  [{mode}] frozen={n_frozen/1e6:.1f}M  trainable={n_train/1e6:.1f}M params")
    return model


class FreezeTrainer(DefaultTrainer):
    """Extends DefaultTrainer with (a) empty-annotation data loader and (b) parameter freezing."""

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_optimizer(cls, cfg, model):
        if _FREEZE_MODE is not None:
            apply_freeze(model, _FREEZE_MODE)
        return super().build_optimizer(cfg, model)


UNLEARN_DATASET = "unlearn"

def _register_unlearn():
    if UNLEARN_DATASET in DatasetCatalog.list():
        return  # already registered
    with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
        coco = json.load(f)
    dicts = [
        {"file_name": str(Path(UNLEARN_DIR)/im["file_name"]),
         "height":    im["height"], "width": im["width"],
         "image_id":  im["id"],     "annotations": []}
        for im in coco["images"]
    ]
    DatasetCatalog.register(UNLEARN_DATASET, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered '{UNLEARN_DATASET}': {len(dicts)} images")

_register_unlearn()

## Reference proxy: poisoned model (no fine-tuning)

In [ ]:
cfg_ref = build_cfg_base(POISONED_WEIGHTS, None, score_thresh=CONF_THRESH, seed=SEED)
pred_ref = DefaultPredictor(cfg_ref)

supp_ref  = supp_score(pred_ref, UNLEARN_DIR)
pres_ref  = pres_score(pred_ref, str(PROBES_DIR))
prx_ref, g_ref, l_ref = proxy(supp_ref, pres_ref)

print(f"Reference (poisoned model):")
print(f"  suppression_score = {supp_ref:.4f}  (target: drive toward 0)")
print(f"  preservation_score = {pres_ref:.4f}  (target: keep high)")
print(f"  proxy = {prx_ref:.4f}  (suppression_gain={g_ref:.4f}, preservation_loss={l_ref:.4f})")

## Train all 3 freeze modes and compute proxy scores

In [ ]:
import gc
global _FREEZE_MODE

results = {
    "reference": {
        "freeze_mode": "reference (no training)",
        "suppression": supp_ref, "preservation": pres_ref,
        "proxy": prx_ref, "supp_gain": g_ref, "pres_loss": l_ref,
    }
}

for mode in ["heads_only", "all_but_cls_head", "lastFPN_plus_heads"]:
    print(f"\n{'='*60}")
    print(f"Training freeze_mode = {mode}")
    print(f"{'='*60}")

    out_dir = OUT_DIR / f"model_{mode}"
    cfg_m = build_cfg_base(POISONED_WEIGHTS, out_dir, score_thresh=CONF_THRESH, seed=SEED)
    cfg_m.DATASETS.TRAIN   = (UNLEARN_DATASET,)
    cfg_m.DATASETS.TEST    = ()
    cfg_m.DATALOADER.NUM_WORKERS = 2
    cfg_m.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
    cfg_m.SOLVER.BASE_LR         = UNLEARN_LR
    cfg_m.SOLVER.MAX_ITER        = UNLEARN_ITERS
    cfg_m.SOLVER.STEPS           = []

    _FREEZE_MODE = mode
    trainer = FreezeTrainer(cfg_m)
    trainer.resume_or_load(resume=False)
    trainer.train()
    _FREEZE_MODE = None

    # Load trained model for inference
    cfg_infer = build_cfg_base(
        out_dir / "model_final.pth", None, score_thresh=CONF_THRESH, seed=SEED
    )
    pred_m = DefaultPredictor(cfg_infer)

    supp_m  = supp_score(pred_m, UNLEARN_DIR)
    pres_m  = pres_score(pred_m, str(PROBES_DIR))
    prx_m, g_m, l_m = proxy(supp_m, pres_m)

    print(f"  suppression_score  = {supp_m:.4f}  (suppression_gain={g_m:.4f})")
    print(f"  preservation_score = {pres_m:.4f}  (preservation_loss={l_m:.4f})")
    print(f"  PROXY              = {prx_m:.4f}")

    results[mode] = {
        "freeze_mode": mode, "suppression": supp_m, "preservation": pres_m,
        "proxy": prx_m, "supp_gain": g_m, "pres_loss": l_m,
    }

    # Free GPU memory between runs
    del trainer, pred_m
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll modes complete.")

## Results table

In [ ]:
print("\n=== PHASE 2 TASK 2.1 — BACKBONE FREEZE RESULTS ===")
print(f"{'Mode':<30} {'Supp_now':>9} {'Supp_gain':>10} {'Pres_now':>9} {'Pres_loss':>10} {'PROXY':>8}")
print("-" * 80)
for key, r in results.items():
    print(f"{r['freeze_mode']:<30} {r['suppression']:>9.4f} {r['supp_gain']:>10.4f} "
          f"{r['preservation']:>9.4f} {r['pres_loss']:>10.4f} {r['proxy']:>8.4f}")

# Identify best mode
modes_only = {k:v for k,v in results.items() if k != 'reference'}
best_mode  = max(modes_only, key=lambda k: modes_only[k]['proxy'])
print(f"\nBest freeze mode: {best_mode}  (proxy={results[best_mode]['proxy']:.4f})")

# Save results
with open(OUT_DIR / "nb10_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved to {OUT_DIR}/nb10_results.json")

## Inference with best model → submission.csv

In [ ]:
best_weights = OUT_DIR / f"model_{best_mode}" / "model_final.pth"
cfg_best = build_cfg_base(best_weights, None, score_thresh=CONF_THRESH, seed=SEED)
pred_best = DefaultPredictor(cfg_best)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} test images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_for_inference(img_path)
    out = pred_best(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts = []
    for (x1,y1,x2,y2),s in zip(boxes, scores):
        x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
        x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
        w,h = max(0.0,x2-x1), max(0.0,y2-y1)
        if w==0 or h==0: continue
        parts.extend([f"{float(s):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"])
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"Wrote submission.csv  ({len(submission)} rows)")

# Validate
assert len(submission) == 2000
assert list(submission.columns) == ["id","image_id","prediction_string"]
empty = (submission["prediction_string"].str.strip().fillna("")=="").sum()
total_dets = submission["prediction_string"].apply(
    lambda s: len(str(s).split())//5 if isinstance(s,str) and str(s).strip() else 0
).sum()
print(f"  Empty rows: {empty}, Total dets: {total_dets}")
print("✓ Submission valid")